In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "src").is_dir() and (candidate / "project.ipynb").is_file():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pyspark.sql import SparkSession
from src.dataOI import DataOI

SEED = 42

# True: rebuild from the raw CSV through cleaning, the split, and every embedding, then save those results.
# False: skip that work and load the saved train, validation, and test embedding frames from data/processed/embeddings.
REBUILD = False
analytics_name = "random_forest"
selected_embedding = "glove"
embedding_name = selected_embedding

# Initial unweighted run; later experiments can override Spark RF parameters.
feature_cols = [
    "embedding", "condition_vec", "days", "usefulCount_log"
]
rf_params = {
    "numTrees": 200,
    "maxDepth": 16,
    "seed": 42,
}

# `spark.driver.memory` is read only when the JVM starts. Restart the kernel after changing it.
spark = (
    SparkSession.builder 
    .master("local[*]")
    .appName("DrugReviewClassification")
    .config("spark.driver.memory", "4g")
    .config("spark.default.parallelism", "4")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark_context = spark.sparkContext
spark_context.setLogLevel("ERROR")
data_oi = DataOI()


In [2]:
from src.cleaning import data_cleaning
from src.text_features import text_feature_engineering
from src.encoder import encoder
from src.split import split_and_weight

if REBUILD:
    df_raw = data_oi.load_raw_data()
    df_cleaned = data_cleaning(df_raw)
    df_cleaned.cache().count()
    df_engineered = text_feature_engineering(df_cleaned,True)
    df_engineered.cache().count()
    df_train, df_valid, df_test = split_and_weight(df_engineered, SEED)
    df_train, df_valid, df_test = encoder(df_train, df_valid, df_test)
    data_oi.save_split_data(df_train, df_valid, df_test)


In [3]:
from src.embeddings import EmbeddingModel

if REBUILD:
    embedding_model = EmbeddingModel(df_train, df_valid, df_test)
    for name, train_function in EmbeddingModel.train_registry.items():
        print(f"Embedding Model `{name}` Training")
        train_function(embedding_model)
    train_embedding_dict = {}
    valid_embedding_dict = {}
    test_embedding_dict = {}
    for name, predict_function in EmbeddingModel.predict_registry.items():
        print(f"Embedding Model `{name}` Running")
        train_embedding_dict[name], valid_embedding_dict[name], test_embedding_dict[name] = predict_function(embedding_model)
        data_oi.save_embedding_data(
            name,
            train_embedding_dict[name],
            valid_embedding_dict[name],
            test_embedding_dict[name],
        )

else:
    train_embedding_dict, valid_embedding_dict, test_embedding_dict = data_oi.load_embedding_data()

# Restrict downstream training to the representation selected above.
for split_name, frames in [("train", train_embedding_dict), ("valid", valid_embedding_dict), ("test", test_embedding_dict)]:
    if selected_embedding not in frames:
        raise KeyError(f"Missing {selected_embedding!r} embedding in {split_name}")
train_embedding_dict = {selected_embedding: train_embedding_dict[selected_embedding]}
valid_embedding_dict = {selected_embedding: valid_embedding_dict[selected_embedding]}
test_embedding_dict = {selected_embedding: test_embedding_dict[selected_embedding]}
print("Selected embedding:", selected_embedding)


Selected embedding: glove


In [4]:
from src.analytics import AnalyticsModel

# Use analytics_name and model settings from initialization.
analytics_models = {}
for embedding_name in train_embedding_dict.keys():
    analytics_models[embedding_name] = AnalyticsModel(
        embedding_name,
        train_embedding_dict[embedding_name],
        valid_embedding_dict[embedding_name],
        test_embedding_dict[embedding_name],
    )


In [5]:
train_pred_dict = {}
valid_pred_dict = {}
test_pred_dict = {}

for embedding_name in train_embedding_dict.keys():
    print(f"Embedding Model `{embedding_name}` Analytics Model `{analytics_name}` Running")
    print("Feature columns:", feature_cols)
    print("Random forest parameters:", rf_params)
    # Format: `{analytics}__{embedding}`. Either name may contain a single underscore.

    model_name = f"{analytics_name}__{embedding_name}"
    train_pred_dict[model_name], valid_pred_dict[model_name], test_pred_dict[model_name] = AnalyticsModel.registry[analytics_name](
        analytics_models[embedding_name],
        feature_cols=None,
        rf_params=None,
    )
    data_oi.save_prediction_data(
        model_name,
        train_pred_dict[model_name],
        valid_pred_dict[model_name],
        test_pred_dict[model_name],
    )


Embedding Model `glove` Analytics Model `random_forest` Running
Feature columns: ['embedding', 'condition_vec', 'days', 'usefulCount_log']
Random forest parameters: {'numTrees': 200, 'maxDepth': 16, 'seed': 42}
Random forest feature columns: ['embedding', 'condition_vec', 'days', 'usefulCount_log']
Random forest effective parameters: {'numTrees': 100, 'maxDepth': 12, 'seed': 42, 'minInstancesPerNode': 5, 'maxMemoryInMB': 64, 'weightCol': 'weight'}
Random forest model saved: E:\DSAI4205\durg project\Drug-Review-Dataset\src\analytics\models\random_forest__glove


In [6]:
from src.evaluation import Scorer

# Score only predictions produced by this experiment.
scorer_train = Scorer(train_pred_dict)
print("Training Data:")
scorer_train.show_metrics(truncate = 80)

scorer_valid = Scorer(valid_pred_dict)
print("Validation Data:")
scorer_valid.show_metrics(truncate = 80)

scorer_test = Scorer(test_pred_dict)
print("Testing Data:")
scorer_test.show_metrics(truncate = 80)

Training Data:
+---------------+--------------+----------+------------------+-----------+----------+----------+
|Embedding_Model|Analytic_Model|  accuracy|weighted_precision|weighted_f1|       mae|      rmse|
+---------------+--------------+----------+------------------+-----------+----------+----------+
|          glove| random_forest|0.71444730|        0.71975950| 0.71036853|1.13255784|2.61196820|
+---------------+--------------+----------+------------------+-----------+----------+----------+

Validation Data:
+---------------+--------------+----------+------------------+-----------+----------+----------+
|Embedding_Model|Analytic_Model|  accuracy|weighted_precision|weighted_f1|       mae|      rmse|
+---------------+--------------+----------+------------------+-----------+----------+----------+
|          glove| random_forest|0.33820759|        0.31278045| 0.31993438|2.18040728|3.39693369|
+---------------+--------------+----------+------------------+-----------+----------+---------